# animation of scale invariance of the power law
# used in presentations

In [1]:
%load_ext jupyter_black

In [5]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.collections import LineCollection
from src.config.paths import FIGURES_DIR

# Power law parameters
alpha = 1.5
window_ratio = 10
fps = 50
duration_s = 20
n_frames = fps * duration_s

# Zoom from [1, 10] to [100, 1000] and back (peak speed ~0.3 decades/s). log10(x_min) follows a cosine, so the zoom accelerates and decelerates smoothly and the GIF loops without a jump.
phase = np.arange(n_frames) / n_frames
x_min_values = 10 ** (1.0 * (1 - np.cos(2 * np.pi * phase)))


def smoothstep(t):
    t = np.clip(t, 0, 1)
    return t * t * (3 - 2 * t)


def tick_opacities(lo, hi, fade_from, fade_to):
    """Opacity of every tick value in [lo, hi] on the nested 1-5-10 grid (..., 1, 5, 10, 50, ...).

    A level with spacing s fades in as its spacing in axes units s / (hi - lo) grows from fade_from to fade_to, so ticks blend in and out continuously instead of jumping.
    Values shared by several levels keep the opacity of the coarsest (most visible) one.
    """
    span = hi - lo
    decade = int(np.floor(np.log10(span)))
    opacities = {}
    # Levels are visited fine -> coarse, so later (larger) opacities overwrite earlier ones.
    for exponent in range(decade - 3, decade + 2):
        for mantissa in (1, 5):
            spacing = mantissa * 10.0**exponent
            opacity = smoothstep((spacing / span - fade_from) / (fade_to - fade_from))
            if opacity == 0:
                continue
            for n in range(int(np.ceil(lo / spacing)), int(np.floor(hi / spacing)) + 1):
                opacities[float(f"{n * spacing:.6g}")] = opacity
    return opacities


def format_x(value):
    return f"{value:,.6g}"


def format_y(value):
    exponent = int(np.floor(np.log10(value) + 1e-9))
    mantissa = f"{value / 10**exponent:.2g}"
    if exponent == 0:
        return mantissa
    return rf"$10^{{{exponent}}}$" if mantissa == "1" else rf"${mantissa}\times10^{{{exponent}}}$"


fig = plt.figure(figsize=(7.2, 5.2), facecolor="white")
# Fixed axes rectangle (no tight_layout): tick labels change every frame, the layout must not.
ax = fig.add_axes([0.17, 0.13, 0.77, 0.77], facecolor="white")
ax.set_xticks([])
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_color("#777777")
    spine.set_linewidth(0.8)

ax.set_title(rf"Power law:  $y = x^{{-{alpha}}}$", fontsize=13, pad=12, color="black")
ax.set_xlabel(r"$x$", fontsize=11, color="black")
ax.set_ylabel(r"$y$", fontsize=11, color="black", rotation=0)
ax.xaxis.set_label_coords(0.5, -0.1)
ax.yaxis.set_label_coords(-0.19, 0.5)

# Grid lines and tick marks live in axes coordinates, one segment per tick value.
grid_x = LineCollection([], linewidths=0.7, transform=ax.transAxes, zorder=0)
grid_y = LineCollection([], linewidths=0.7, transform=ax.transAxes, zorder=0)
tickmarks_x = LineCollection([], linewidths=0.8, transform=ax.transAxes, clip_on=False)
tickmarks_y = LineCollection([], linewidths=0.8, transform=ax.transAxes, clip_on=False)
for collection in (grid_x, grid_y, tickmarks_x, tickmarks_y):
    ax.add_collection(collection)

(curve,) = ax.plot([], [], color="black", linewidth=2.2, zorder=3)
(scale_marker,) = ax.plot([], [], "o", color="#b4a58f", markersize=5, zorder=4)
window_text = ax.text(0.04, 0.94, "", transform=ax.transAxes, ha="left", va="top", fontsize=9, color="#777777")
tick_labels = []

grey = np.array([0.45, 0.45, 0.45])
tick_length = 0.012  # axes units


def draw_axis(lo, hi, axis):
    """Grid, tick marks and labels for one axis. axis is "x" or "y"."""
    # Fade windows in axes units: faint grid lines appear at ~1% spacing, labels only once they have room.
    label_fade = {"x": (0.075, 0.13), "y": (0.055, 0.11)}[axis]
    grid = tick_opacities(lo, hi, fade_from=0.012, fade_to=0.06)
    labelled = tick_opacities(lo, hi, *label_fade)

    positions = (np.array(list(grid)) - lo) / (hi - lo)  # axes coordinates
    # Labelled lines are darker; both terms vary continuously with the zoom.
    grid_alpha = 0.18 * np.array(list(grid.values())) + 0.3 * np.array([labelled.get(v, 0) for v in grid])
    grid_colors = np.column_stack([np.tile(grey, (len(grid), 1)), grid_alpha])

    label_values = np.array(list(labelled))
    label_positions = (label_values - lo) / (hi - lo)
    # Labels also fade out near the axis ends, so they do not pop in at the edge or overlap the other axis.
    label_alpha = np.array(list(labelled.values())) * smoothstep(label_positions / 0.05) * smoothstep((1 - label_positions) / 0.05)
    label_colors = np.column_stack([np.tile([0.27, 0.27, 0.27], (len(labelled), 1)), label_alpha])

    if axis == "x":
        grid_x.set_segments([[(u, 0), (u, 1)] for u in positions])
        grid_x.set_colors(grid_colors)
        tickmarks_x.set_segments([[(u, 0), (u, -tick_length)] for u in label_positions])
        tickmarks_x.set_colors(label_colors)
        for u, value, color in zip(label_positions, label_values, label_colors):
            tick_labels.append(ax.text(u, -0.025, format_x(value), transform=ax.transAxes, ha="center", va="top", fontsize=9, color=color))
    else:
        grid_y.set_segments([[(0, v), (1, v)] for v in positions])
        grid_y.set_colors(grid_colors)
        tickmarks_y.set_segments([[(0, v), (-tick_length, v)] for v in label_positions])
        tickmarks_y.set_colors(label_colors)
        for v, value, color in zip(label_positions, label_values, label_colors):
            tick_labels.append(ax.text(-0.02, v, format_y(value), transform=ax.transAxes, ha="right", va="center", fontsize=9, color=color))


def format_window(value):
    return f"{value:.2f}" if value < 10 else f"{value:.1f}" if value < 100 else f"{value:,.0f}"


def update(frame):
    x_min = x_min_values[frame]
    x_max = window_ratio * x_min

    # Evaluate the original power law in the current window.
    x_window = np.linspace(x_min, x_max, 300)
    curve.set_data(x_window, x_window ** (-alpha))
    scale_marker.set_data([x_min], [x_min ** (-alpha)])

    # Linear axes: both limits change with the scale being inspected.
    # Proportional padding preserves the same displayed shape.
    x_lo, x_hi = 0.9 * x_min, 1.1 * x_max
    y_lo, y_hi = 0.85 * x_max ** (-alpha), 1.15 * x_min ** (-alpha)
    ax.set_xlim(x_lo, x_hi)
    ax.set_ylim(y_lo, y_hi)

    while tick_labels:
        tick_labels.pop().remove()
    draw_axis(x_lo, x_hi, "x")
    draw_axis(y_lo, y_hi, "y")

    window_text.set_text(f"window:  [{format_window(x_min)}, {format_window(x_max)}]")

anim = FuncAnimation(fig, update, frames=n_frames, interval=1000 / fps)
output_path = FIGURES_DIR / "power_law_scale_freeness.gif"
# FFMpegWriter builds a per-GIF palette (palettegen / paletteuse), which keeps the grey fades clean.
anim.save(output_path, writer=FFMpegWriter(fps=fps), dpi=100)
plt.close(fig)

print(f"Saved {n_frames} frames ({duration_s} s at {fps} fps) to: {output_path}")

Saved 1000 frames (20 s at 50 fps) to: C:\Users\andre\OneDrive\Education\4_Tuebingen\Comp_Neuroscience\3_4_Lab_rotations\1_Anna_Levina_critical_networks\repository\results\figures\power_law_scale_freeness.gif
